# Autodata 3.45 — analiză statică în Colab (fără Azure)

Acest notebook citește inventarul și pachetele ZIP deja salvate în Google Drive. Nu pornește și nu importă executabile, DLL, activatoare, keygen, scripturi CMD/BAT/VBS sau fișiere din arhivă ca programe. Nu încearcă ocolirea blocărilor Google Drive. Pachetul 0016, blocat anterior de Drive, este exclus implicit.

Analiza include verificarea fiecărui fragment SHA256, reconstruirea temporară a fișierelor de date, identificarea semnăturilor cunoscute, analiza descriptivă a fișierelor IDF (fără a pretinde că formatul necunoscut este decodat), structură XML, metadate pentru documente și dimensiuni pentru imagini standard.

**De pe telefon:** Runtime → Run all. Autorizează Google Drive o singură dată. Rezultatele se salvează în folderul Autodata_3_45_ANALYSIS, creat în folderul-sursă. Fișierele reconstruite rămân temporar în Colab, nu sunt reîncărcate în Drive.

**Atenție:** rularea poate descărca până la aproximativ 2,3 GB în Colab. Dacă un ZIP este refuzat de Drive, va fi marcat ca indisponibil și nu va fi accesat prin altă metodă. Nu este garantată decodarea formatului IDF.


In [ ]:
from google.colab import auth
auth.authenticate_user()
import collections, csv, hashlib, io, json, math, os, re, struct, time, zipfile
from pathlib import Path, PurePosixPath
from xml.etree import ElementTree as ET
from html.parser import HTMLParser
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from googleapiclient.errors import HttpError

SOURCE_FOLDER = "1XqpDaYI5zkdUXY85wLSvpsklatT0P0Bd"
RESULTS_FOLDER_NAME = "Autodata_3_45_ANALYSIS"
WORK = Path("/content/autodata_static_v1")
CACHE = WORK / "packages"
DATA = WORK / "safe_data"
REPORTS = WORK / "reports"
for folder in (CACHE, DATA, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive = build("drive", "v3", credentials=creds, cache_discovery=False)

def list_children(folder_id):
    items, token = [], None
    while True:
        result = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType,size)",
            pageSize=1000, pageToken=token).execute()
        items.extend(result.get("files", []))
        token = result.get("nextPageToken")
        if not token:
            return items

source_items = {item["name"]: item for item in list_children(SOURCE_FOLDER)}
required = ["INVENTAR.json", "INVENTAR.txt", "listare_7z.txt", "TRANSFER_FINALIZAT.txt"]
missing = [name for name in required if name not in source_items]
if missing:
    raise RuntimeError("Lipsesc fișierele de control din Drive: " + ", ".join(missing))

def read_small(name, max_bytes=1_000_000):
    info = source_items[name]
    if int(info.get("size", 0)) > max_bytes:
        raise ValueError("Fișier de control neașteptat de mare: " + name)
    blob = drive.files().get_media(fileId=info["id"]).execute()
    if len(blob) > max_bytes:
        raise ValueError("Dimensiune excesivă: " + name)
    return blob

manifest = json.loads(read_small("INVENTAR.json").decode("utf-8-sig"))
confirmation = read_small("TRANSFER_FINALIZAT.txt").decode("utf-8-sig")
print("INVENTAR:", manifest.get("file_count"), "fișiere; pachete:", manifest.get("package_count"))
print("Confirmare Drive:", confirmation.strip().replace("\n", " | "))

In [ ]:
# Auditul manifestului înainte de descărcarea vreunui ZIP.
SAFE_EXTENSIONS = {".idf", ".xml", ".txt", ".dat", ".htm", ".html", ".pdf",
                   ".chm", ".bmp", ".png", ".jpg", ".jpeg", ".gif"}
BLOCKED_BY_DRIVE_PREVIOUSLY = {"autodata_part_0016.zip"}
MAX_TOTAL_BYTES = 3_000_000_000
MAX_FILE_BYTES = 900_000_000
MAX_CHUNK_BYTES = 70_000_000

def safe_relative(name):
    if not isinstance(name, str) or not name or "\\" in name or "\x00" in name:
        raise ValueError("Nume de fișier nesigur")
    p = PurePosixPath(name)
    if p.is_absolute() or any(x in ("", ".", "..") for x in p.parts) or ":" in p.parts[0]:
        raise ValueError("Cale nesigură: " + name[:120])
    return p

files = manifest.get("files")
if not isinstance(files, list) or len(files) != 139:
    raise ValueError("Manifestul nu conține cele 139 fișiere așteptate.")
if manifest.get("file_count") != len(files) or manifest.get("package_count") != 16:
    raise ValueError("Număr de fișiere/pachete neconcordant.")
if manifest.get("sha256") not in confirmation:
    raise ValueError("SHA256 al arhivei nu corespunde confirmării.")
seen_paths, seen_entries, package_names = set(), set(), set()
total_bytes, total_chunks = 0, 0
for item in files:
    p = str(safe_relative(item["path"]))
    if p in seen_paths:
        raise ValueError("Cale duplicată: " + p)
    seen_paths.add(p)
    size = item["size_bytes"]
    if not isinstance(size, int) or not 0 <= size <= MAX_FILE_BYTES:
        raise ValueError("Dimensiune invalidă: " + p)
    chunks = item.get("chunks")
    if not isinstance(chunks, list) or not chunks:
        raise ValueError("Fragmente lipsă: " + p)
    if sum(c["bytes"] for c in chunks) != size:
        raise ValueError("Dimensiunea fragmentelor nu corespunde: " + p)
    for chunk in chunks:
        package, entry, count, digest = (chunk[k] for k in ("package", "entry", "bytes", "sha256"))
        safe_relative(entry)
        if not re.fullmatch(r"autodata_part_\d{4}\.zip", package):
            raise ValueError("Pachet invalid: " + package)
        if not isinstance(count, int) or not 0 <= count <= MAX_CHUNK_BYTES:
            raise ValueError("Fragment prea mare: " + p)
        if not isinstance(digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", digest):
            raise ValueError("SHA256 invalid: " + p)
        if (package, entry) in seen_entries:
            raise ValueError("Intrare ZIP duplicată: " + entry)
        seen_entries.add((package, entry))
        package_names.add(package)
        total_chunks += 1
    total_bytes += size
if total_bytes != manifest.get("total_extracted_bytes") or total_bytes > MAX_TOTAL_BYTES:
    raise ValueError("Dimensiunea totală a inventarului nu corespunde.")
if len(package_names) != 16 or any(p not in source_items for p in package_names):
    raise ValueError("Lipsesc pachete ZIP sau numărul este incorect.")
ext_counts = collections.Counter(PurePosixPath(f["path"]).suffix.lower() for f in files)
safe_files = [f for f in files if PurePosixPath(f["path"]).suffix.lower() in SAFE_EXTENSIONS]
needed_packages = {c["package"] for f in safe_files for c in f["chunks"]}
manifest_audit = {
    "archive_sha256": manifest["sha256"], "manifest_files": len(files),
    "chunks": total_chunks, "packages": len(package_names), "total_bytes": total_bytes,
    "extensions": dict(sorted(ext_counts.items())),
    "eligible_data_files": len(safe_files),
    "excluded_packages_known_blocked": sorted(BLOCKED_BY_DRIVE_PREVIOUSLY),
    "status": "MANIFEST_VALIDATED"
}
(REPORTS / "manifest_audit.json").write_text(
    json.dumps(manifest_audit, indent=2, ensure_ascii=False), encoding="utf-8")
print("MANIFEST_VALIDATED=PASS", total_chunks, "fragmente", len(safe_files), "fișiere de date")

In [ ]:
# Folosește numai API-ul Google Drive obișnuit; nu setează acknowledgeAbuse și nu încearcă linkuri alternative.
def reason_from_http_error(error):
    try:
        body = json.loads(error.content.decode("utf-8", errors="replace"))
        e = body.get("error", {})
        reasons = [x.get("reason", "") for x in e.get("errors", [])]
        return reasons[0] if reasons else "unknown"
    except Exception:
        return "unknown"

def download_zip(name):
    if name in BLOCKED_BY_DRIVE_PREVIOUSLY:
        return None, "SKIPPED_KNOWN_DRIVE_BLOCK"
    info = source_items[name]
    target = CACHE / name
    expected = int(info["size"])
    if target.exists() and target.stat().st_size == expected:
        try:
            with zipfile.ZipFile(target) as z:
                bad = z.testzip()
                if bad is None:
                    return target, "CACHED_ZIP_TEST_PASS"
        except (OSError, zipfile.BadZipFile):
            pass
        target.unlink()
    for attempt in range(4):
        partial = target.with_suffix(".partial")
        try:
            with partial.open("wb") as output:
                request = drive.files().get_media(fileId=info["id"])
                downloader = MediaIoBaseDownload(output, request, chunksize=8 * 1024 * 1024)
                done = False
                while not done:
                    _, done = downloader.next_chunk(num_retries=2)
            if partial.stat().st_size != expected:
                raise IOError("ZIP incomplet: " + name)
            partial.replace(target)
            with zipfile.ZipFile(target) as z:
                bad = z.testzip()
                if bad is not None:
                    raise zipfile.BadZipFile("CRC eșuat: " + bad)
            return target, "ZIP_TEST_PASS"
        except HttpError as exc:
            partial.unlink(missing_ok=True)
            reason = reason_from_http_error(exc)
            if reason == "cannotDownloadAbusiveFile":
                return None, "BLOCKED_BY_GOOGLE_DRIVE"
            if exc.resp.status in (401, 403, 404):
                return None, f"HTTP_{exc.resp.status}_{reason}"
            if exc.resp.status not in (408, 429, 500, 502, 503, 504):
                return None, f"HTTP_{exc.resp.status}_{reason}"
            if attempt == 3:
                return None, f"RETRY_EXHAUSTED_{exc.resp.status}"
        except (OSError, IOError, zipfile.BadZipFile) as exc:
            partial.unlink(missing_ok=True)
            target.unlink(missing_ok=True)
            if attempt == 3:
                return None, "ZIP_INVALID_OR_NETWORK_ERROR_" + type(exc).__name__
        time.sleep(min(2 ** attempt, 10))
    return None, "UNAVAILABLE"

package_status, available = {}, {}
for i, name in enumerate(sorted(needed_packages), 1):
    path, status = download_zip(name)
    package_status[name] = status
    if path is not None:
        available[name] = path
    print(f"[{i}/{len(needed_packages)}] {name}: {status}")
(REPORTS / "package_status.json").write_text(
    json.dumps(package_status, indent=2, ensure_ascii=False), encoding="utf-8")
print("Pachete disponibile:", len(available), "din", len(needed_packages))

In [ ]:
# Citește strict intrările aprobate de manifest; nu folosește ZipFile.extract() / extractall().
def sample_entropy(data):
    if not data:
        return 0.0
    freq = collections.Counter(data)
    return round(-sum((n / len(data)) * math.log2(n / len(data)) for n in freq.values()), 4)

def signature(data):
    checks = [
        (b"SQLite format 3\x00", "SQLite"),
        (b"%PDF-", "PDF"), (b"\x89PNG\r\n\x1a\n", "PNG"),
        (b"BM", "BMP"), (b"GIF87a", "GIF"), (b"GIF89a", "GIF"),
        (b"PK\x03\x04", "ZIP"), (b"MZ", "PE/EXE"),
        (b"\x7fELF", "ELF"), (b"Rar!\x1a\x07", "RAR")
    ]
    return next((label for magic, label in checks if data.startswith(magic)), "UNKNOWN")

class Tags(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.tags = collections.Counter()
    def handle_starttag(self, tag, attrs):
        self.tags[tag] += 1

def static_metadata(path, extension):
    with path.open("rb") as stream:
        prefix = stream.read(65536)
    result = {
        "first_16_bytes_hex": prefix[:16].hex(),
        "recognized_signature": signature(prefix),
        "prefix_entropy_bits_per_byte": sample_entropy(prefix),
        "prefix_printable_ascii_fraction": round(
            sum(32 <= b < 127 or b in (9, 10, 13) for b in prefix) / max(len(prefix), 1), 4)
    }
    if extension == ".idf":
        result["interpretation"] = "FORMAT_NOT_DECODED; metadata only"
    elif extension == ".xml":
        if path.stat().st_size > 5_000_000:
            result["xml"] = "SKIPPED_SIZE_LIMIT"
        else:
            data = path.read_bytes()
            if b"<!DOCTYPE" in data.upper() or b"<!ENTITY" in data.upper():
                result["xml"] = "SKIPPED_DTD_OR_ENTITY"
            else:
                try:
                    root = ET.fromstring(data)
                    counts = collections.Counter(el.tag.split("}")[-1] for el in root.iter())
                    result["xml"] = {
                        "root_tag": root.tag.split("}")[-1],
                        "elements": sum(counts.values()),
                        "top_element_tags": counts.most_common(12)
                    }
                except ET.ParseError as exc:
                    result["xml"] = "PARSE_ERROR: " + str(exc)[:140]
    elif extension in (".htm", ".html") and path.stat().st_size < 2_000_000:
        parser = Tags()
        parser.feed(path.read_text(encoding="utf-8", errors="replace"))
        result["html_tag_counts"] = parser.tags.most_common(12)
    elif extension in (".bmp", ".png"):
        if extension == ".png" and prefix.startswith(b"\x89PNG\r\n\x1a\n") and len(prefix) >= 24:
            result["image_dimensions"] = list(struct.unpack(">II", prefix[16:24]))
        elif extension == ".bmp" and prefix.startswith(b"BM") and len(prefix) >= 26:
            result["image_dimensions"] = list(struct.unpack("<ii", prefix[18:26]))
    elif extension in (".pdf", ".chm"):
        result["document"] = "HEADER_ONLY_NO_ACTIVE_CONTENT"
    elif extension in (".txt", ".dat"):
        result["text_metadata"] = "NO_CONTENT_EXPORTED"
    return result

def rebuild_one(item):
    rel = safe_relative(item["path"])
    ext = rel.suffix.lower()
    if ext not in SAFE_EXTENSIONS:
        return {"path": str(rel), "status": "SKIPPED_EXECUTABLE_OR_UNSUPPORTED_TYPE"}
    required = [c["package"] for c in item["chunks"]]
    absent = sorted(set(required) - set(available))
    if absent:
        return {"path": str(rel), "status": "UNAVAILABLE_PACKAGE", "packages": absent}
    dest = DATA.joinpath(*rel.parts)
    dest.parent.mkdir(parents=True, exist_ok=True)
    file_digest = hashlib.sha256()
    count_total = 0
    try:
        with dest.open("wb") as output:
            for chunk in item["chunks"]:
                with zipfile.ZipFile(available[chunk["package"]]) as z:
                    info = z.getinfo(chunk["entry"])
                    if info.is_dir() or info.file_size != chunk["bytes"] or info.file_size > MAX_CHUNK_BYTES:
                        raise ValueError("Intrare ZIP invalidă: " + chunk["entry"])
                    digest = hashlib.sha256()
                    copied = 0
                    with z.open(info) as src:
                        while True:
                            block = src.read(1024 * 1024)
                            if not block:
                                break
                            output.write(block)
                            digest.update(block)
                            file_digest.update(block)
                            copied += len(block)
                    if copied != chunk["bytes"] or digest.hexdigest().lower() != chunk["sha256"].lower():
                        raise ValueError("SHA256 sau dimensiune fragment incorectă")
                    count_total += copied
        if count_total != item["size_bytes"]:
            raise ValueError("Dimensiune finală incorectă")
        return {
            "path": str(rel), "status": "VERIFIED", "bytes": count_total,
            "sha256": file_digest.hexdigest(), "extension": ext,
            "static": static_metadata(dest, ext)
        }
    except Exception as exc:
        dest.unlink(missing_ok=True)
        return {"path": str(rel), "status": "ERROR", "error": type(exc).__name__ + ": " + str(exc)[:160]}

results = []
for i, item in enumerate(files, 1):
    result = rebuild_one(item)
    results.append(result)
    if i % 20 == 0 or i == len(files):
        print(f"Analizate {i}/{len(files)} | VERIFIED={sum(r['status']=='VERIFIED' for r in results)}")
(REPORTS / "file_report.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
with (REPORTS / "file_report.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["path", "status", "bytes", "sha256", "extension", "recognized_signature", "interpretation"])
    writer.writeheader()
    for item in results:
        static = item.get("static", {})
        writer.writerow({
            "path": item["path"], "status": item["status"], "bytes": item.get("bytes", ""),
            "sha256": item.get("sha256", ""), "extension": item.get("extension", ""),
            "recognized_signature": static.get("recognized_signature", ""),
            "interpretation": static.get("interpretation", "")
        })
counts = collections.Counter(x["status"] for x in results)
summary = [
    "# Autodata 3.45 — raport de analiză statică",
    "", "Arhivă SHA256: " + manifest["sha256"],
    "Fișiere inventariate: " + str(len(files)),
    "Fragmente inventariate: " + str(total_chunks),
    "Fișiere verificate și analizate: " + str(counts.get("VERIFIED", 0)),
    "Fișiere indisponibile: " + str(counts.get("UNAVAILABLE_PACKAGE", 0)),
    "Fișiere excluse după tip: " + str(counts.get("SKIPPED_EXECUTABLE_OR_UNSUPPORTED_TYPE", 0)),
    "Erori de verificare: " + str(counts.get("ERROR", 0)),
    "", "Pachete:", *[f"- {name}: {status}" for name, status in sorted(package_status.items())],
    "", "IMPORTANT: Formatul IDF nu este considerat decodat; doar semnături și metadate.",
    "Nu au fost executate programe sau scripturi din arhivă."
]
(REPORTS / "summary.md").write_text("\n".join(summary) + "\n", encoding="utf-8")
print("STATIC_ANALYSIS_COMPLETE", dict(counts))

In [ ]:
# Persistă NUMAI rapoarte și metadate în Drive; fișierele brute rămân temporare în Colab.
existing_results = [
    item for item in list_children(SOURCE_FOLDER)
    if item["name"] == RESULTS_FOLDER_NAME and item["mimeType"] == "application/vnd.google-apps.folder"
]
if existing_results:
    output_id = existing_results[0]["id"]
else:
    output_id = drive.files().create(
        body={"name": RESULTS_FOLDER_NAME, "mimeType": "application/vnd.google-apps.folder",
              "parents": [SOURCE_FOLDER]}, fields="id").execute()["id"]
existing = {item["name"]: item for item in list_children(output_id)}
for path in sorted(REPORTS.iterdir()):
    if not path.is_file():
        continue
    media = MediaFileUpload(str(path), mimetype="application/json" if path.suffix == ".json" else
                            "text/csv" if path.suffix == ".csv" else "text/markdown",
                            resumable=True)
    if path.name in existing:
        drive.files().update(fileId=existing[path.name]["id"], media_body=media).execute()
    else:
        drive.files().create(body={"name": path.name, "parents": [output_id]},
                             media_body=media, fields="id").execute()
    print("SALVAT:", path.name)
print("RAPOARTE_DRIVE=PASS")
print("FOLDER_REZULTATE=https://drive.google.com/drive/folders/" + output_id)